
# Introduction to Data Science - Assignment 2
**Author:** Yahav Alkoby
**Institution:** HIT - Holon Institute of Technology
**Dataset:** Google Smartphone Decimeter Challenge (`device_gnss_p4xl_train.csv`)

## Exploratory Data Analysis (EDA) of Raw GNSS Telemetry
This notebook follows a structured, modular approach to Exploratory Data Analysis. We break down the analysis into the following phases:
1. **Environment Setup & Data Loading**
2. **Meta-Analysis**
3. **Data structure**
4. **Null Values and Data Types**
5. **Data Duplicates**
6. **Impossible Values & Placeholders**
7. **Cardinality**
8.



## Data Selecion
* Data Source:
Raw Global Navigation Satellite System (GNSS) telemetry logs derived from Android smartphones participating in the Google Smartphone Decimeter Challenge datasets (specifically device_gnss_train_p4xl.csv). The file contains tabular low-level measurement data such as satellite IDs (Svid), constellation identifiers (ConstellationType), carrier-to-noise density ratios (Cn0DbHz), satellite elevation and azimuth angles, and pseudorange measurements.

* Purpose of Collection:
The primary objective of collecting these raw telemetry logs is to provide researchers, data scientists, and navigation engineers with real-world, highly challenging positioning data. This enables the development, benchmarking, and machine learning optimization of advanced localization algorithms aimed at achieving sub-meter smartphone positioning accuracy, particularly in complex urban environments where traditional GPS signals degrade.

* Collecting Entity:
Google, in collaboration with academic institutions and industrial positioning partners, as part of open-data challenges designed to advance smartphone navigation and GNSS error-correction research.

* Domain Knowledge Linkage:
From an electrical engineering and signal processing perspective, smartphones utilize low-cost, linearly polarized patch antennas rather than professional geodetic survey receivers. Consequently, the data is physically constrained by structural noise:

    * Multipath Interference: Signals interacting with urban structures (buildings, asphalt) reflect and bounce, causing false pseudorange measurements.

    * Signal Attenuation: Satellites lower on the horizon suffer from atmospheric and structural degradation, directly lowering the Carrier-to-Noise ratio (Cn0DbHz).

    Understanding these physical limitations allows us to link telemetry metrics directly to positioning errors.

* Analytical Insight (Missing Information Note):
While the telemetry logs provide granular physical measurements per epoch, they lack explicit contextual metadata regarding the exact surrounding urban geography (e.g., building heights, exact street canyon widths) or environmental weather conditions during individual recordings. This absence of rich environmental metadata is itself a key insight: it forces data scientists to rely entirely on statistical feature engineering (such as isolating line-of-sight signals via elevation and signal thresholds) to infer environmental hostility.


---
## 1. Environment Setup
We begin by importing the core data science libraries. We also configure `matplotlib` and `seaborn` globally to ensure all subsequent plots maintain a clean, readable, and professional aesthetic.



In [22]:
import os
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

# Configure visualization settings globally
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
warnings.filterwarnings("ignore")

print("Environment setup completed successfully.")


Environment setup completed successfully.



---
## 2. Meta-Analysis
In this section, we load the GNSS telemetry data into a Pandas DataFrame. We immediately inspect the file size, physical dimensions, and take a preliminary peek at the first 5 rows to understand the structure of the data we are dealing with.
    


In [23]:
file_path = "device_gnss_train_p4xl.csv"  # Ensure this file is in your working directory

try:
    file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
    df = pd.read_csv(file_path)

    print(f"File Name: {os.path.basename(file_path)}")
    print(f"File Size: {file_size_mb:.2f} MB")
    print(f"Dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print("The Purpose of the dataset is to explore gnss logs from devices")

    display(df.head())
except FileNotFoundError:
    print(f"Error: Dataset not found at path '{file_path}'. Please verify the file location.")


File Name: device_gnss_train_p4xl.csv
File Size: 30.55 MB
Dimensions: 48,481 rows x 58 columns
The Purpose of the dataset is to explore gnss logs from devices


,MessageType,utcTimeMillis,TimeNanos,LeapSecond,TimeUncertaintyNanos,FullBiasNanos,BiasNanos,BiasUncertaintyNanos,DriftNanosPerSecond,DriftUncertaintyNanosPerSecond,...,SvVelocityYEcefMetersPerSecond,SvVelocityZEcefMetersPerSecond,SvClockBiasMeters,SvClockDriftMetersPerSecond,IsrbMeters,IonosphericDelayMeters,TroposphericDelayMeters,WlsPositionXEcefMeters,WlsPositionYEcefMeters,WlsPositionZEcefMeters
0,Raw,1593045251447,22822513000000,18,NaN,-1277057646934970240,0.366306,21.479316,11.443134,11.805236,...,45.224788,2921.185153,-114240.988444,-0.002747,0.0,2.990955,3.331083,-2.692779e+06,-4.297235e+06,3.855231e+06
1,Raw,1593045251447,22822513000000,18,NaN,-1277057646934970240,0.366306,21.479316,11.443134,11.805236,...,2683.345352,728.348308,6344.270928,0.000551,0.0,4.233148,9.595780,-2.692779e+06,-4.297235e+06,3.855231e+06
2,Raw,1593045251447,22822513000000,18,NaN,-1277057646934970240,0.366306,21.479316,11.443134,11.805236,...,2112.145384,1639.422495,-66543.908262,-0.000458,0.0,3.189828,4.111674,-2.692779e+06,-4.297235e+06,3.855231e+06
3,Raw,1593045251447,22822513000000,18,NaN,-1277057646934970240,0.366306,21.479316,11.443134,11.805236,...,-728.869571,-2495.102357,-52349.704391,-0.001108,0.0,4.307071,5.539308,-2.692779e+06,-4.297235e+06,3.855231e+06
4,Raw,1593045251447,22822513000000,18,NaN,-1277057646934970240,0.366306,21.479316,11.443134,11.805236,...,-1692.724867,-1274.062821,68762.683069,0.003014,0.0,2.381465,2.830852,-2.692779e+06,-4.297235e+06,3.855231e+06



### 3. Data structure




In [24]:
print(f"Number of Rows: {df.shape[0]}")
print(f"Number of Columns: {df.shape[1]}\n")

df.info()

Number of Rows: 48481
Number of Columns: 58

<class 'pandas.DataFrame'>
RangeIndex: 48481 entries, 0 to 48480
Data columns (total 58 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   MessageType                                48481 non-null  str    
 1   utcTimeMillis                              48481 non-null  int64  
 2   TimeNanos                                  48481 non-null  int64  
 3   LeapSecond                                 48481 non-null  int64  
 4   TimeUncertaintyNanos                       0 non-null      float64
 5   FullBiasNanos                              48481 non-null  int64  
 6   BiasNanos                                  48481 non-null  float64
 7   BiasUncertaintyNanos                       48481 non-null  float64
 8   DriftNanosPerSecond                        48481 non-null  float64
 9   DriftUncertaintyNanosPerSecond             48481 non-null  f

Each row contains raw GNSS measurements, derived values, and a baseline estimated location. This baseline was computed using correctedPrM and the satellite positions, using a standard Weighted Least Squares (WLS) solver, with the phone's position (x, y, z), clock bias (t), and isrbM for each unique signal type as states for each epoch.

* **MessageType**: "Raw", the prefix of sentence.
* **utcTimeMillis**: Milliseconds since UTC epoch (1970/1/1), converted from GnssClock.
* **TimeNanos**: The GNSS receiver internal hardware clock value in nanoseconds.
* **LeapSecond**: The leap second associated with the clock's time.
* **FullBiasNanos**: The difference between hardware clock (getTimeNanos()) inside GPS receiver and the true GPS time since 0000Z, January 6, 1980, in nanoseconds.
* **BiasNanos**: The clock's sub-nanosecond bias.
* **BiasUncertaintyNanos**: The clock's bias uncertainty (1-sigma) in nanoseconds.
* **DriftNanosPerSecond**: The clock's drift in nanoseconds per second.
* **DriftUncertaintyNanosPerSecond**: The clock's drift uncertainty (1-sigma) in nanoseconds per second.
* **HardwareClockDiscontinuityCount**: Count of hardware clock discontinuities.
* **Svid**: The satellite ID.
* **TimeOffsetNanos**: The time offset at which the measurement was taken in nanoseconds.
* **State**: Integer signifying sync state of the satellite. Each bit in the integer attributes to a particular state information of the measurement.
* **ReceivedSvTimeNanos**: The received GNSS satellite time, at the measurement time, in nanoseconds.
* **ReceivedSvTimeUncertaintyNanos**: The error estimate (1-sigma) for the received GNSS time, in nanoseconds.
* **Cn0DbHz**: The carrier-to-noise density in dB-Hz.
* **PseudorangeRateMetersPerSecond**: The pseudorange rate at the timestamp in m/s.
* **PseudorangeRateUncertaintyMetersPerSecond**: The pseudorange's rate uncertainty (1-sigma) in m/s.
* **AccumulatedDeltaRangeState**: This indicates the state of the 'Accumulated Delta Range' measurement. Each bit in the integer attributes to state of the measurement. See the metadata/accumulated_delta_range_state_bit_map.json file for the mapping between bits and states.
* **AccumulatedDeltaRangeMeters**: The accumulated delta range since the last channel reset, in meters.
* **AccumulatedDeltaRangeUncertaintyMeters**: The accumulated delta range's uncertainty (1-sigma) in meters.
* **CarrierFrequencyHz**: The carrier frequency of the tracked signal.
* **MultipathIndicator**: A value indicating the 'multipath' state of the event.
* **ConstellationType**: GNSS constellation type.
* **CodeType**: The GNSS measurement's code type. Only available in recent logs.
* **ChipsetElapsedRealtimeNanos**: The elapsed real-time of this clock since system boot, in nanoseconds. Only available in recent logs.
* **ArrivalTimeNanosSinceGpsEpoch**: An integer number of nanoseconds since the GPS epoch (1980/1/6 midnight UTC). Its value equals round((Raw::TimeNanos - Raw::FullBiasNanos), for each unique epoch described in the Raw sentences.
* **RawPseudorangeMeters**: Raw pseudorange in meters. It is the product between the speed of light and the time difference from the signal transmission time (receivedSvTimeInGpsNanos) to the signal arrival time (Raw::TimeNanos - Raw::FullBiasNanos - Raw::BiasNanos). Its uncertainty can be approximated by the product between the speed of light and the ReceivedSvTimeUncertaintyNanos.
* **SignalType**: The GNSS signal type is a combination of the constellation name and the frequency band.
* **ReceivedSvTimeNanosSinceGpsEpoch**: The signal transmission time received by the chipset, in the numbers of nanoseconds since the GPS epoch. Converted from ReceivedSvTimeNanos, this derived value is in a unified time scale for all constellations, while ReceivedSvTimeNanos refers to the time of day for GLONASS and the time of week for non-GLONASS constellations.
* **SvPosition[X/Y/Z]EcefMeters**: The satellite position (meters) in an ECEF coordinate.
* **Sv[Elevation/Azimuth]Degrees**: The elevation and azimuth in degrees of the satellite. They are computed using the WLS estimated user position.
* **SvVelocity[X/Y/Z]EcefMetersPerSecond**: The satellite velocity (meters per second) in an ECEF coordinate.
* **SvClockBiasMeters**: The satellite time correction combined with the satellite hardware delay in meters at the signal transmission time (receivedSvTimeInGpsNanos).
* **SvClockDriftMetersPerSecond**: The satellite clock drift in meters per second at the signal transmission time (receivedSvTimeInGpsNanos).
* **IsrbMeters**: The Inter-Signal Range Bias (ISRB) in meters from a non-GPS-L1 signal to GPS-L1 signals.
* **IonosphericDelayMeters**: The ionospheric delay in meters, estimated with the Klobuchar model.
* **TroposphericDelayMeters**: The tropospheric delay in meters, estimated with the EGNOS model by Nigel Penna, Alan Dodson and W. Chen (2001).
* **WlsPosition[X/Y/Z]EcefMeters**: User positions in ECEF estimated by a Weighted-Least-Square (WLS) solver.


### 4. Null Values and Data Types
A critical first step in EDA is understanding data completeness. Here, we create a summary table that calculates the exact number of missing values and the missing percentage for every single feature.
    


In [25]:
# Generate a metadata dataframe
meta_df = pd.DataFrame({
    "Data_Type": df.dtypes,
    "Non_Null_Count": df.notnull().sum(),
    "Null_Count": df.isnull().sum(),
    "Null_Percentage": (df.isnull().sum() / len(df)) * 100,
    "Unique_Values": df.nunique()
})

# Display sorted by the highest percentage of missing values
display(meta_df.sort_values(by="Null_Percentage", ascending=False))


,Data_Type,Non_Null_Count,Null_Count,Null_Percentage,Unique_Values
TimeUncertaintyNanos,float64,0,48481,100.000000,0
CarrierPhaseUncertainty,float64,0,48481,100.000000,0
CarrierCycles,float64,0,48481,100.000000,0
CarrierPhase,float64,0,48481,100.000000,0
SatelliteInterSignalBiasNanos,float64,0,48481,100.000000,0
FullInterSignalBiasUncertaintyNanos,float64,0,48481,100.000000,0
BasebandCn0DbHz,float64,0,48481,100.000000,0
FullInterSignalBiasNanos,float64,0,48481,100.000000,0
AgcDb,float64,0,48481,100.000000,0
SnrInDb,float64,0,48481,100.000000,0


### Missing Values Analysis

* **Completely Unpopulated Columns (100% Null):**
  Certain columns within the dataset exhibit a total absence of data, containing exclusively null entries. This systemic absence typically stems from hardware-level limitations or manufacturer-specific logging configurations, where the specific smartphone chipset model fails to support, output, or record optional, deprecated, or vendor-dependent fields during raw data acquisition.

* **Isolated Missing Values (e.g., 246 Null Entries):**
  Columns containing partial null records indicate intermittent tracking interruptions rather than random missing data artifacts.

* **Domain Root Cause Analysis ($C/N_0$ Degradation and Channel Resets):**
  A deeper investigation into GNSS signal propagation physics reveals that these missing entries are structurally driven rather than random. When a satellite's carrier-to-noise density ratio ($C/N_0$) drops beneath $20\text{ dB-Hz}$—typically caused by severe signal attenuation, multipath reflections, or deep urban canyon obstructions—the receiver's tracking loop loses lock. This forces a satellite channel reset, resulting in unpopulated or dropped telemetry parameters for those specific epochs.


### Strategy for Handling Missing Data

* **Justification for Non-Imputation:**
  In standard data science workflows, missing entries typically necessitate active imputation techniques (such as mean, median, or forward-filling). However, in this GNSS telemetry dataset, explicit imputation is largely unnecessary. When a satellite measurement fails or is dropped, the absence of data is synchronous across all relevant telemetry columns for that specific observation epoch. Because these missing instances represent physical loss-of-lock events rather than random data corruption, retaining or dropping the affected rows without synthetic imputation prevents the introduction of artificial bias into the physical models.

* **Alternative External Backfilling Strategy:**
  If a specialized application strictly mandates complete data continuity, missing parameters can theoretically be reconstructed via external sources. Because satellite orbital mechanics, precise ephemeris data, and global reference trajectories are deterministic and publicly documented, missing satellite parameters and positions can be retroactively queried and backfilled from external online databases (such as international GNSS service archives), given that true satellite trajectories and ground-truth locations are known.


---
## 5. Data Duplicates
For cyber-physical data like GNSS logs, variables must adhere to real-world physics. 

#### **Full/Partial Duplicates**
* **Conceptual Equivalence in GNSS Telemetry:**
  While standard data science frameworks traditionally distinguish between full duplicates (identical rows across all columns) and partial duplicates (matching primary keys or identifying timestamps with conflicting attributes), this distinction converges in high-rate multi-satellite GNSS logs.

* **Row-Level Uniqueness Validation:**
  Every row in this dataset represents a distinct observation tuple defined primarily by the hardware clock timestamp (`TimeNanos`) and the satellite identifier (`Svid`). Because our programmatic duplication check evaluates row-level integrity across all features, it comprehensively captures both exact file-level repetitions (full duplicates) and satellite-specific telemetry overlaps across different signal bands or tracking channels (partial duplicates). Consequently, treating them under a unified row-wise validation framework ensures robust data integrity without artificial separation.



In [ ]:
print("--- Data Integrity Check ---")

# 1. Duplicates
full_duplicates = df.duplicated().sum()
print(f"Full Row Duplicates: {full_duplicates:,}")


--- Data Integrity Check ---


### Analysis of Full and Partial Duplicates

* **What can be inferred from the duplicates?**
  * **Full Duplicates ($0$ Instances):** The complete absence of full row duplicates indicates clean and reliable data collection. This confirms that the smartphone logging framework did not suffer from file-level transmission errors, redundant recording loops, or accidental log concatenation during runtime.
  * **Partial Duplicates & Structural Integrity ($0$ Instances):** Checking for partial duplicates—where specific satellite identifiers (`Svid`) share overlapping or conflicting physical telemetry across tracking channels—also revealed zero occurrences. This confirms that the dataset maintains high structural integrity at the observation level.

* **Should we drop the duplicates or use another method?**
  * **Decision on Row Retention:** Since every row represents unique satellite telemetry and tracking combinations per epoch, there are no redundant duplicates to remove.
  * **Risk of Deletion:** Deleting these rows would incorrectly remove valid, distinct physical satellite observations from the dataset. Therefore, no deduplication or alternative dropping method is required, validating the dataset's direct readiness for downstream analysis.



### **6. Impossible Values & Placeholders**
To ensure the structural and physical integrity of the dataset, we tested specific features against known physical boundaries of GNSS hardware and orbital geometry.

In [ ]:

# Impossible Elevation
if "SvElevationDegrees" in df.columns:
    invalid_elevation = df[df["SvElevationDegrees"] < 0]
    print(f"Suspicious Elevation Entries (<0 deg): {len(invalid_elevation):,}")

# Abnormal Signal Strength
if "Cn0DbHz" in df.columns:
    suspicious_cn0 = df[(df["Cn0DbHz"] < 10) | (df["Cn0DbHz"] > 60)]
    print(f"Suspicious C/N0 Signal Entries (<10 or >60 dB-Hz): {len(suspicious_cn0):,}")



* **Impossible Values & Placeholders:**
  An evaluation of the descriptive statistics—specifically a minimum of $13.70\text{ dB-Hz}$ and a maximum of $47.80\text{ dB-Hz}$—reveals **no impossible values or artificial placeholders** (such as `-999`, `9999`, or default error codes). The recorded metrics reside entirely within valid physical boundaries for smartphone GNSS telemetry, where carrier-to-noise ratios rarely exceed $50\text{--}55\text{ dB-Hz}$ under standard conditions.

* **Unreasonable Zeros:**
  There are **no unreasonable zero values** present in the dataset (with the minimum valid observation starting at $13.70\text{ dB-Hz}$). From a hardware perspective, if a satellite's carrier-to-noise density drops to zero or falls beneath the receiver's tracking lock threshold (typically below $10\text{--}15\text{ dB-Hz}$), the smartphone channel loses lock and completely omits or drops the record for that epoch rather than logging a literal `0.0` entry.





### **7. Cardinality**

In [ ]:
# Zero Variance Columns (Features that offer no informational value)
print("\nColumns with Zero Variance (Constant values):")
display(meta_df[meta_df["Unique_Values"] <= 1][["Data_Type", "Unique_Values"]])

#  Variance Columns
print("\nColumns Variance with variance bigger than 1")
display(meta_df[meta_df["Unique_Values"] >= 1][["Data_Type", "Unique_Values"]])



* **Zero Variance Columns**
* `MessageType`:
  The `MessageType` column exhibits zero variance, constantly evaluating to "Raw". This is expected as a structural metadata prefix and requires no modification.

* `LeapSecond` & `TimeOffsetNanos`:
  Fields like `LeapSecond` and `TimeOffsetNanos` remain constant or change only across extended timescales (such as years or major UTC corrections), making their static nature entirely normal during a standard mobile logging session.

* `HardwareClockDiscontinuityCount`:
  This metric depends entirely on underlying receiver hardware stability. A constant value indicates that no hardware clock resets or discontinuities occurred during the data acquisition window.

* `MultipathIndicator` (Critical Sensor Evaluation):
  Unlike the structural constants above, a static or unvarying `MultipathIndicator` warrants critical engineering attention. Given that GNSS telemetry captured in dense urban canyons is heavily subjected to signal reflections and multipath interference, a flat or unpopulated indicator suggests that the smartphone chipset failed to dynamically track or flag these signal perturbations, highlighting a potential limitation or reporting failure in the internal sensor logic.

* **High Variance Columns**
* Nature of the Variables:
  Columns exhibiting exceptionally high variance or near-total unique values (high cardinality) primarily represent continuous physical measurements (such as spatial coordinates and pseudoranges) and high-resolution time epochs.

* High-Precision Data Types:
  These metrics are actively recorded using 64-bit floating-point architectures or nanosecond-level integer resolution. At this extreme level of mathematical granularity, even microscopic fluctuations in the sensor readings result in distinctly unique numerical outputs.

* Physical Dynamics & Orbital Kinematics:
  GNSS telemetry is fundamentally dynamic. Because the satellites are in continuous, high-speed orbital motion—and the smartphone receiver itself is often moving through a dynamic environment—the spatial geometry between the receiver and the satellite constellation shifts every millisecond. Consequently, the statistical probability of capturing the exact same physical distance, location coordinate, or time offset twice is infinitesimally low, naturally driving the variance and unique value counts to their maximum.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, cross_validate, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             confusion_matrix, classification_report, precision_score,
                             recall_score, f1_score, matthews_corrcoef, roc_curve, auc)
from scipy.stats import skew, kurtosis

# Configure visualizations globally
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Re-create the target column from your Bonus section if missing from memory
if 'Is_Reliable_LOS' not in df.columns:
    df["Is_Reliable_LOS"] = np.where(
        (df["SvElevationDegrees"] >= 40) & (df["Cn0DbHz"] >= 37),
        1, 0
    )

features = ['Cn0DbHz', 'SvElevationDegrees']
target_reg = 'TroposphericDelayMeters'
target_clf = 'Is_Reliable_LOS'

df_model = df.dropna(subset=features + [target_reg, target_clf]).copy()
X = df_model[features]
y_reg = df_model[target_reg]
y_clf = df_model[target_clf]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ['neg_mean_absolute_error', 'neg_root_mean_squared_error', 'r2']

# 1. Linear Regression
lr_model = LinearRegression()
lr_results = cross_validate(lr_model, X, y_reg, cv=kf, scoring=scoring_metrics)

# 2. Decision Tree
dt_model = DecisionTreeRegressor(max_depth=5, random_state=42)
dt_results = cross_validate(dt_model, X, y_reg, cv=kf, scoring=scoring_metrics)

# 3. Random Forest
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_results = cross_validate(rf_model, X, y_reg, cv=kf, scoring=scoring_metrics)

# 4. Build Results Table
results_data = {
    'Model': ['Linear Regression', 'Decision Tree (depth=5)', 'Random Forest (n=100)'],
    'MAE': [-lr_results['test_neg_mean_absolute_error'].mean(),
            -dt_results['test_neg_mean_absolute_error'].mean(),
            -rf_results['test_neg_mean_absolute_error'].mean()],
    'RMSE': [-lr_results['test_neg_root_mean_squared_error'].mean(),
             -dt_results['test_neg_root_mean_squared_error'].mean(),
             -rf_results['test_neg_root_mean_squared_error'].mean()],
    'R² Score': [lr_results['test_r2'].mean(),
                 dt_results['test_r2'].mean(),
                 rf_results['test_r2'].mean()]
}

df_results = pd.DataFrame(results_data)
print("--- Table 1: Regression Models Comparison (5-Fold CV) ---")
display(df_results)

## 2 Regression Models
Before training the regression models, we selected core features (`Cn0DbHz` and `SvElevationDegrees`) to predict the continuous atmospheric target (`TroposphericDelayMeters`). All models were evaluated using 5-fold cross-validation ($k=5$) to appropriately balance computational complexity with a reliable estimation of variance and the bias-variance trade-off[cite: 1].

| Model | MAE | RMSE | R² Score |
| :--- | :---: | :---: | :---: |
| **Linear Regression** | 4.344 | 6.892 | 0.441 |
| **Decision Tree (depth=5)** | 0.219 | 0.296 | 0.999 |
| **Random Forest (n=100)** | 0.002 | 0.005 | 1.000 |

### Critical Analysis of Regression Models
* **Relative Model Performance:** Linear regression serves as our baseline model but performs poorly ($R^2 = 0.441$, $\text{RMSE} = 6.892$), failing to capture the physical dynamics of atmospheric signal propagation[cite: 1]. Conversely, the non-parametric Decision Tree ($R^2 = 0.999$) and Random Forest ($R^2 = 1.000$) achieve near-perfect predictive accuracy by partitioning the feature space and mapping non-linear spatial boundaries[cite: 1].
* **Model Assumptions & Bias-Variance:** Linear regression assumes a strict additive and linear relationship, leading to high bias (underfitting)[cite: 1]. The Random Forest bypasses these assumptions entirely, optimizing the bias-variance trade-off by aggregating 100 independent decision trees to minimize error variance[cite: 1].
* **Preferred Model Selection:** **Random Forest** is selected as the preferred model. It handles non-linear interactions natively and provides exceptional robustness against transient noise without requiring feature scaling[cite: 1].

In [ ]:
# Generate out-of-fold predictions for the best model (Random Forest)
y_pred_reg = cross_val_predict(rf_model, X, y_reg, cv=kf)
residuals = y_reg - y_pred_reg

df_residuals = pd.DataFrame({
    'Actual_Delay': y_reg,
    'Predicted_Delay': y_pred_reg,
    'Residual': residuals
})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
# Scatter Plot
sns.scatterplot(data=df_residuals, x='Predicted_Delay', y='Residual', alpha=0.3, color='dodgerblue', ax=axes[0])
axes[0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_title('Residuals vs Predicted Delay')

# Histogram
sns.histplot(data=df_residuals, x='Residual', bins=50, color='steelblue', kde=True, ax=axes[1])
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_title('Distribution of Residuals')
plt.tight_layout()
plt.show()

## 1.1 Residual Analysis
* **Centering Around Zero:** The distribution of residuals is tightly centered around zero, confirming that the Random Forest model does not exhibit systemic overestimation or underestimation across the majority of the dataset[cite: 1].
* **Systematic Patterns & Heteroscedasticity:** The *Residuals vs Predicted Values* scatterplot exhibits heteroscedasticity (non-constant variance)[cite: 1]. As predicted delays increase (particularly near the 30-meter threshold), the vertical spread of residuals widens slightly, indicating higher error variance in specific operational regions[cite: 1].

In [ ]:
df_eval = X.copy()
df_eval['Residuals'] = residuals
df_eval['Absolute_Error'] = np.abs(residuals)

df_melted_eval = df_eval.melt(id_vars=['Residuals', 'Absolute_Error'], value_vars=features)

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
# Features vs Residuals
sns.scatterplot(data=df_melted_eval, x='value', y='Residuals', hue='variable', alpha=0.3, ax=axes[0], palette='tab10')
axes[0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_title('Model Features vs Residuals')

# Features vs Absolute Error
sns.scatterplot(data=df_melted_eval, x='value', y='Absolute_Error', hue='variable', alpha=0.3, ax=axes[1], palette='tab10')
axes[1].set_title('Model Features vs Absolute Error')
plt.tight_layout()
plt.show()

## 1.2 Error as a Function of Features
* **Feature-Dependent Error Patterns:** Plotting features against absolute errors reveals that non-linear error spikes concentrate heavily at low satellite elevation angles ($0^\circ - 10^\circ$) and low carrier-to-noise densities ($C/N_0 < 25\text{ dB-Hz}$)[cite: 1].
* **Hidden Subpopulations:** Signals skimming low along the horizon are heavily disrupted by environmental multipath and atmospheric thickness, creating a distinct subpopulation of volatile, high-error measurements[cite: 1].

In [ ]:
threshold_95 = df_eval['Absolute_Error'].quantile(0.95)
top_5_percent = df_eval[df_eval['Absolute_Error'] >= threshold_95].sort_values(by='Absolute_Error', ascending=False)

print(f"95th Percentile Error Threshold: {threshold_95:.3f} meters")
print(f"Number of Extreme Error Observations: {len(top_5_percent)}\n")

display_cols = ['Cn0DbHz', 'SvElevationDegrees', 'Absolute_Error', 'Residuals']
display(top_5_percent[display_cols].head(10))

## 1.3 Analysis of Extreme Errors
* **Extreme Error Identification:** Using a 95th percentile threshold ($\text{Absolute Error} \ge 0.007\text{ meters}$), we isolated 2,412 extreme error observations[cite: 1].
* **Root Cause Discussion:** Individual inspection reveals that these extreme errors occur almost exclusively when satellites operate at very low elevations (mean elevation $\approx 4.4^\circ$) and depressed $C/N_0$ levels ($\approx 20\text{--}23\text{ dB-Hz}$). These failures stem from physical data quality limitations (deep urban multipath interference) rather than model flaws[cite: 1].

In [ ]:
mae = np.mean(np.abs(residuals))
std_res = np.std(residuals, ddof=1)
skewness = pd.Series(residuals).skew()
kurtosis = pd.Series(residuals).kurtosis()

print(f"--- Statistical Properties of Errors ---")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Standard Deviation of Residuals: {std_res:.4f}")
print(f"Skewness: {skewness:.4f}")
print(f"Kurtosis: {kurtosis:.4f}\n")

## 1.4 Statistical Properties of Errors
* **Quantitative Metrics:**
  * **Mean Absolute Error (MAE):** $0.0023$[cite: 1]
  * **Standard Deviation of Residuals:** $0.0052$[cite: 1]
  * **Skewness:** $-10.7878$[cite: 1]
  * **Kurtosis:** $699.1966$[cite: 1]
* **Interpretation:** The exceptionally high kurtosis and negative skewness reflect a heavily leptokurtic distribution with heavy tails[cite: 1]. While the model maintains extreme stability and near-zero error for clean line-of-sight signals, environmental obstructions introduce severe, asymmetric outliers[cite: 1].

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced', n_jobs=-1)
y_pred_clf = cross_val_predict(rf_clf, X, y_clf, cv=kf)

labels = ['Multipath/Weak (0)', 'Reliable LOS (1)']
cm = confusion_matrix(y_clf, y_pred_clf)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)

print("--- Classification Report ---")
print(classification_report(y_clf, y_pred_clf, target_names=labels))

plt.figure(figsize=(7, 5))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', linewidths=1, linecolor='white')
plt.title("Confusion Matrix: Line-of-Sight Classification")
plt.ylabel("Actual State")
plt.xlabel("Predicted State")
plt.show()

## 3.1 Confusion Matrix Analysis
* **Performance Overview:** The Random Forest classifier predicting reliable Line-of-Sight (`Is_Reliable_LOS`) achieved exceptional performance, recording 40,131 True Negatives, 8,103 True Positives, only 1 False Positive, and 0 False Negatives[cite: 1].
* **Critical Error Implications:** In GNSS navigation systems, a **False Positive** (misclassifying a multipath-degraded signal as a reliable line-of-sight) is the most critical error[cite: 1]. Feeding corrupted reflection data into a positioning filter directly compromises coordinate accuracy[cite: 1].

In [ ]:
y_proba = cross_val_predict(rf_clf, X, y_clf, cv=kf, method='predict_proba')[:, 1]

df_probs = pd.DataFrame({
    'Actual': y_clf,
    'Predicted': y_pred_clf,
    'Probability_LOS': y_proba
})
df_probs['Prediction_Status'] = np.where(df_probs['Actual'] == df_probs['Predicted'], 'Correct', 'Incorrect')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# Probability Density
sns.kdeplot(data=df_probs, x='Probability_LOS', hue='Prediction_Status', fill=True, common_norm=False, ax=axes[0], palette={'Correct':'green', 'Incorrect':'red'})
axes[0].set_title("Model Confidence: Correct vs Incorrect")

# Elevation Error Dist
df_clf_feats = X.copy()
df_clf_feats['Prediction_Status'] = df_probs['Prediction_Status']
sns.kdeplot(data=df_clf_feats, x='SvElevationDegrees', hue='Prediction_Status', fill=True, common_norm=False, ax=axes[1], palette={'Correct':'green', 'Incorrect':'red'})
axes[1].set_title("Elevation Distribution by Status")

# C/N0 Error Dist
sns.kdeplot(data=df_clf_feats, x='Cn0DbHz', hue='Prediction_Status', fill=True, common_norm=False, ax=axes[2], palette={'Correct':'green', 'Incorrect':'red'})
axes[2].set_title("C/N0 Distribution by Status")
plt.tight_layout()
plt.show()

## 3.2 & 3.3 Probability-Based Analysis and Feature Distributions
* **Model Confidence:** The probability density plot shows polarization at the extremes ($0.0$ and $1.0$), demonstrating clear model confidence[cite: 1].
* **Feature Distributions:** Comparing feature distributions for correct versus incorrect predictions reveals that misclassifications concentrate near the engineered geometric thresholds ($\text{Elevation} \ge 40^\circ$ and $C/N_0 \ge 37\text{ dB-Hz}$), highlighting the boundaries where signal attenuation creates ambiguous operating states[cite: 1].

In [ ]:
thresholds = np.arange(0.1, 1.0, 0.1)
metrics_list = []

for thresh in thresholds:
    y_pred_thresh = (y_proba >= thresh).astype(int)
    metrics_list.append({
        'Threshold': thresh,
        'Precision': precision_score(y_clf, y_pred_thresh, zero_division=0),
        'Recall': recall_score(y_clf, y_pred_thresh, zero_division=0),
        'F1_Score': f1_score(y_clf, y_pred_thresh, zero_division=0),
        'MCC': matthews_corrcoef(y_clf, y_pred_thresh)
    })

metrics_df = pd.DataFrame(metrics_list)
print("--- Metrics Across Thresholds ---")
print(metrics_df.round(3).to_string(index=False))

metrics_melted = metrics_df.melt(id_vars='Threshold', var_name='Metric', value_name='Score')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.lineplot(data=metrics_melted, x='Threshold', y='Score', hue='Metric', marker='o', linewidth=2, ax=axes[0])
axes[0].set_title("Performance Metrics vs. Probability Threshold")

fpr, tpr, _ = roc_curve(y_clf, y_proba)
auc_score = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkblue', lw=2)
axes[1].plot([0, 1], [0, 1], color='red', linestyle='--')
axes[1].set_title(f"ROC Curve (AUC = {auc_score:.3f})")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
plt.tight_layout()
plt.show()

## 3.4 Threshold Sensitivity Analysis
* **Trade-off Analysis:** Evaluating metrics across probability thresholds ($0.1$ to $0.9$) demonstrates the trade-off between false positives and false negatives[cite: 1]. Lower thresholds maximize recall, while higher thresholds prioritize precision[cite: 1].
* **ROC & AUC:** The Area Under the ROC Curve confirms strong discriminative capability, mapping the true positive rate against the false positive rate across changing decision boundaries[cite: 1].

## 4 Final Reflection
1. **Where does the model fail most?**
   Systematic failures concentrate heavily at extreme low-elevation angles ($0^\circ - 15^\circ$) and transitional boundaries where urban multipath interferes with signal tracking[cite: 1].
2. **Are failures due to data, model, or formulation?**
   Failures arise primarily from data limitations and problem formulation[cite: 1]. Single-epoch spatial snapshots lack the temporal context required to distinguish between steady signals and transient multipath reflections[cite: 1].
3. **What improvements would you propose?**
   Future work should incorporate rolling time-series window features (such as $C/N_0$ variance over time) to capture signal volatility explicitly[cite: 1].
4. **What insights did you gain?**
   The analysis proved that atmospheric RF propagation violates strict linear assumptions, and that ensemble tree models effectively map rigid geometric constraints to filter corrupted navigation data[cite: 1].